### MFCC Model

In [1]:
import os
import numpy as np
import librosa
from python_speech_features import mfcc
import matplotlib.pyplot as plt
import samplerate
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from imblearn.over_sampling import SVMSMOTE
import tensorflow.keras as keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix
import seaborn as sns
from keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

In [ ]:

# 去噪（使用二阶的巴特沃夫带通滤波器）
def band_pass_filter(original_signal, order, fc1, fc2, fs):
    b, a = signal.butter(N=order, Wn=[2*fc1/fs, 2*fc2/fs], btype='bandpass')
    new_signal = signal.lfilter(b, a, original_signal)
    return new_signal

# 重采样到2kHz
def resample_audio(audio_data, fs):
    audio_data = samplerate.resample(audio_data, 2000/fs, 'sinc_best').T
    return audio_data

# 数据增强
def add_noise1(x, w=0.004):  # 加噪
    output = x + w * np.random.normal(loc=0, scale=1, size=len(x))
    return output

def add_noise2(x, snr=50):  # 控制信噪比
    P_signal = np.sum(abs(x) ** 2) / len(x)  # 信号功率
    P_noise = P_signal / 10 ** (snr / 10.0)  # 噪声功率
    return x + np.random.randn(len(x)) * np.sqrt(P_noise)

def time_shift(x, shift):  # 波形移动
    return np.roll(x, int(shift))

def time_stretch(x, rate):  # 波形拉伸
    return librosa.effects.time_stretch(x, rate)

# 归一化
def normalize(x):
    return x / np.max(np.abs(x))

# 信号分割
def segment_signal(signal):
    segment_length = 3000  
    segments = []
    
    for i in range(0, len(signal), segment_length):
        segment = signal[i:i + segment_length]
        if len(segment) == segment_length:
            segments.append(segment)
    
    return np.array(segments)

# 提取MFCC特征
def getMFCCMap(y, sr=2000):
    mfcc0 = mfcc(y, sr, numcep=13, winlen=0.025, winstep=0.01)  
    mf1 = librosa.feature.delta(mfcc0)
    mf2 = librosa.feature.delta(mfcc0, order=2)
    mfcc_all = np.hstack((mfcc0, mf1, mf2))
    
    # 3000点音频按默认参数生成约149帧
    mfcc_all = pad_or_trim_mfcc(mfcc_all, target_shape=(149, 39))  
    
    return mfcc_all.reshape(149, 39, 1)  

# 确保音频数据长度一致
def pad_or_trim_mfcc(mfcc_all, target_shape=(149, 39)):
   
    if mfcc_all.shape[0] < target_shape[0]:
        pad_width = [(0, target_shape[0]-mfcc_all.shape[0]), (0,0)]
        return np.pad(mfcc_all, pad_width, mode='constant')
    else:
        return mfcc_all[:target_shape[0], :]

# 初始化特征和标签列表
Mfcc_features = []
Mfcc_labels = []

 # 指定文件夹路径
for i, label in enumerate(['n', 'abn']):
    folder_path = f'two_class_data/{label}/'
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        audio_data, fs = librosa.load(file_path, sr=None)
        audio_data = band_pass_filter(audio_data, 2, 25, 400, fs)
        down_sample_audio_data = resample_audio(audio_data, fs)
        segments = segment_signal(down_sample_audio_data)
        
        # 动态增强策略
        if np.random.rand() > 0.5:
            down_sample_audio_data = add_noise1(down_sample_audio_data)
        if np.random.rand() > 0.5:
            down_sample_audio_data = time_shift(down_sample_audio_data, shift=200)
        
        # 生成多个片段
        for seg in segments:
            data_Mfcc = getMFCCMap(seg, 2000)
            Mfcc_features.append(data_Mfcc)
        Mfcc_labels.extend([i] * len(segments))


In [ ]:
# Mfcc_features = np.save('Mfcc_features.npy',Mfcc_features)
# Mfcc_labels = np.save('Mfcc_labels.npy',Mfcc_labels)

In [2]:
Mfcc_features = np.load("Mfcc_features.npy")
Mfcc_labels = np.load("Mfcc_labels.npy")

In [3]:
Mfcc_labels = np.array(Mfcc_labels)
Mfcc_labels.shape

(50480,)

In [ ]:
import numpy as np

# 将特征和标签列表转换为 NumPy 数组
Mfcc_features = np.array(Mfcc_features)
Mfcc_labels = np.array(Mfcc_labels)

# 检查特征和标签的形状
print("Mfcc_features shape:", Mfcc_features.shape)  
print("Mfcc_labels shape:", Mfcc_labels.shape)      

Mfcc_features shape: (50480, 149, 39, 1)
Mfcc_labels shape: (50480,)


In [4]:
Mfcc_features.shape
Mfcc_labels.shape

(50480,)

In [ ]:
import numpy as np
from sklearn.utils import shuffle

def custom_train_test_split(features, labels, test_size=0.3, random_state=42):
    """
    自定义数据集划分函数
    :param features: 特征数组 (num_samples, ...)
    :param labels: 标签数组 (num_samples,)
    :param test_size: 测试集占总数据的比例
    :param random_state: 随机种子，确保可重复性
    :return: train_features, test_features, train_label, test_label
    """

    features = np.array(features)
    labels = np.array(labels)
    
    # 打乱数据和标签
    features, labels = shuffle(features, labels, random_state=random_state)
    
    # 计算训练集和测试集的大小
    num_samples = len(labels)
    test_size = int(num_samples * test_size)
    train_size = num_samples - test_size
    
    # 划分训练集和测试集
    train_features = features[:train_size]
    train_label = labels[:train_size]
    test_features = features[train_size:]
    test_label = labels[train_size:]
    
    return train_features, test_features, train_label, test_label

train_features, test_features, train_label, test_label = custom_train_test_split(
    Mfcc_features, Mfcc_labels, test_size=0.3, random_state=42
)

print("Train features shape:", train_features.shape)  
print("Train labels shape:", train_label.shape)       
print("Test labels shape:", test_label.shape)         

Train features shape: (35336, 149, 39, 1)
Train labels shape: (35336,)
Test features shape: (15144, 149, 39, 1)
Test labels shape: (15144,)


In [6]:
import numpy as np

def custom_compute_class_weight(labels):
    """
    自定义计算类别权重
    :param labels: 标签数组 (num_samples,)
    :return: 类别权重字典
    """
    # 计算每个类别的样本数量
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    # 计算每个类别的权重
    total_samples = len(labels)
    class_weights = {label: total_samples / (len(unique_labels) * count) for label, count in zip(unique_labels, counts)}
    
    return class_weights

# 使用自定义函数计算类别权重
class_weights = custom_compute_class_weight(train_label)

# 打印类别权重
print("Class weights:", class_weights)

Class weights: {0: 0.6752790093257912, 1: 1.9262974269515918}


In [ ]:
from keras.utils import to_categorical

# 将整数标签转换为 one-hot 编码
train_label = to_categorical(train_label, num_classes=2)
test_label = to_categorical(test_label, num_classes=2)

# 检查转换后的标签形状
print("Train labels shape:", train_label.shape)  
print("Test labels shape:", test_label.shape)   

Train labels shape: (35336, 2)
Test labels shape: (15144, 2)


In [ ]:
# 构建模型
def build_model():
    inputdata = keras.Input(shape=(149, 39, 1))  
    x = keras.layers.BatchNormalization()(inputdata)
    
    # 卷积块
    x = keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputdata)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2,2))(x)
    
    x = keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2,2))(x)
    
    # 时空特征提取
    x = keras.layers.Reshape((-1, 64))(x)
    x = keras.layers.Bidirectional(keras.layers.GRU(64, return_sequences=True))(x)
    x = keras.layers.Bidirectional(keras.layers.GRU(64))(x)
    
    # 分类头
    x = keras.layers.Dropout(0.5)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    outputs = keras.layers.Dense(2, activation='softmax')(x)
    
    # 使用自适应学习率
    model = keras.Model(inputs=inputdata, outputs=outputs)
    optimizer = keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(loss='categorical_crossentropy',  
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model

# 训练模型
num_epochs = 100
model = build_model()

from keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)


history = model.fit(
    train_features,
    train_label,
    validation_split=0.2,
    epochs=num_epochs,
    batch_size=128,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights
)

np.save('test_features.npy', test_features)
np.save('test_label.npy', test_label)

# 评估模型
test_features = np.load("test_features.npy")
test_label = np.load("test_label.npy")
test_predictions = model.predict(test_features)
test_predictions_labels = np.argmax(test_predictions, axis=1)

# 计算混淆矩阵
C2 = confusion_matrix(test_label.argmax(axis=1), test_predictions_labels)
print(C2)

# 绘制混淆矩阵
f, ax = plt.subplots()
sns.heatmap(C2, fmt='g', annot=True, cmap='YlGnBu', cbar=False, ax=ax,
            xticklabels=('normal', 'abnormal'),
            yticklabels=('normal', 'abnormal'))
ax.set_title('Confusion matrix')  
ax.set_xlabel('Predicted')  
ax.set_ylabel('True')  
plt.show()

# 计算准确率
accuracy = np.mean(test_predictions_labels == test_label.argmax(axis=1))
print(f'Test Accuracy: {accuracy:.4f}')

# 绘制损失曲线和准确率曲线
plt.figure(figsize=(14, 5))

# 绘制损失曲线
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# 绘制准确率曲线
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
## 基本的cnn模型 
import numpy as np
import keras
from keras import backend as K
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


print("原始标签分布:", Counter(Mfcc_labels))

train_features, test_features, train_labels, test_labels = train_test_split(
    Mfcc_features, Mfcc_labels,
    test_size=0.3,
    random_state=42,
    stratify=Mfcc_labels
)

print("训练集标签分布:", Counter(train_labels))
print("测试集标签分布:", Counter(test_labels))

# One-hot 编码
y_train = to_categorical(train_labels, num_classes=2)
y_test = to_categorical(test_labels, num_classes=2)


weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(weights))
print("类别权重:", class_weights)

# ======================
# Focal Loss 定义
# ======================
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * K.log(y_pred)
        weight = alpha * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.sum(loss, axis=1)
    return focal_loss_fixed

# ======================
# 模型结构定义
# ======================
def super_simple_cnn_model():
    inputdata = keras.Input(shape=(149, 39, 1))  
    
    x = keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inputdata)
    x = keras.layers.MaxPooling2D((2, 2))(x)

    x = keras.layers.Flatten()(x)
    
    outputs = keras.layers.Dense(2, activation='softmax')(x)
    
    model = keras.Model(inputs=inputdata, outputs=outputs)
    optimizer = keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(loss=focal_loss(gamma=2.0, alpha=0.75),
                  optimizer=optimizer,
                  metrics=['accuracy'])
    
    return model

# ======================
# 模型训练
# ======================
model = build_model()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

history = model.fit(
    train_features, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128,
    class_weight=class_weights,
    # callbacks=[early_stop, reduce_lr]
)

# ======================
# 模型评估
# ======================
test_predictions = model.predict(test_features)
y_pred = np.argmax(test_predictions, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
sns.heatmap(confusion_matrix(y_true, y_pred), fmt='g', annot=True, cmap='YlGnBu', cbar=False,
            xticklabels=['normal', 'abnormal'],
            yticklabels=['normal', 'abnormal'])
plt.title('Confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['normal', 'abnormal']))

print(f'Test Accuracy: {np.mean(y_pred == y_true):.4f}')

# ======================
# 训练曲线
# ======================
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
## 确定的crnn 效果最好
import numpy as np
import keras
from keras import backend as K
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


print("原始标签分布:", Counter(Mfcc_labels))

train_features, test_features, train_labels, test_labels = train_test_split(
    Mfcc_features, Mfcc_labels,
    test_size=0.3,
    random_state=42,
    stratify=Mfcc_labels
)

print("训练集标签分布:", Counter(train_labels))
print("测试集标签分布:", Counter(test_labels))

# One-hot 编码
y_train = to_categorical(train_labels, num_classes=2)
y_test = to_categorical(test_labels, num_classes=2)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(weights))
print("类别权重:", class_weights)


def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = K.epsilon()
        y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * K.log(y_pred)
        weight = alpha * K.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return K.sum(loss, axis=1)
    return focal_loss_fixed

def build_model():
    inputdata = keras.Input(shape=(149, 39, 1))
    x = keras.layers.BatchNormalization()(inputdata)

    x = keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2,2))(x)

    x = keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2,2))(x)

    x = keras.layers.Reshape((-1, 64))(x)
    x = keras.layers.Bidirectional(keras.layers.GRU(64, return_sequences=True, dropout=0.3))(x)
    x = keras.layers.Bidirectional(keras.layers.GRU(64, dropout=0.3))(x)

    x = keras.layers.Dropout(0.5)(x)
    x = keras.layers.Dense(32, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    outputs = keras.layers.Dense(2, activation='softmax')(x)

    model = keras.Model(inputs=inputdata, outputs=outputs)
    optimizer = keras.optimizers.Adam(learning_rate=1e-4)
    model.compile(loss=focal_loss(gamma=2.0, alpha=0.75),
                  optimizer=optimizer,
                  metrics=['accuracy'])
    return model


model = build_model()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

history = model.fit(
    train_features, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=128,
    class_weight=class_weights,
    # callbacks=[early_stop, reduce_lr]
)


test_predictions = model.predict(test_features)
y_pred = np.argmax(test_predictions, axis=1)
y_true = np.argmax(y_test, axis=1)

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
sns.heatmap(confusion_matrix(y_true, y_pred), fmt='g', annot=True, cmap='YlGnBu', cbar=False,
            xticklabels=['normal', 'abnormal'],
            yticklabels=['normal', 'abnormal'])
plt.title('Confusion matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=['normal', 'abnormal']))

print(f'Test Accuracy: {np.mean(y_pred == y_true):.4f}')


plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Curves')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# 引入transformer模块 但未使用focal loss,有待进一步完善,（注：训练时间很长！）

# import numpy as np
# import keras
# from keras import backend as K
# from keras.layers import MultiHeadAttention, LayerNormalization, Dense, Add, Dropout, Reshape
# from keras.models import Sequential 
# from keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from keras.utils import to_categorical
# from sklearn.metrics import confusion_matrix, classification_report
# from sklearn.utils import class_weight
# from sklearn.model_selection import train_test_split
# from collections import Counter
# import matplotlib.pyplot as plt
# import seaborn as sns


# Mfcc_features = np.expand_dims(Mfcc_features, axis=-1)
# print("Mfcc_features reshaped shape:", Mfcc_features.shape) 
# print("原始标签分布:", Counter(Mfcc_labels))

# train_features, test_features, train_labels, test_labels = train_test_split(
#     Mfcc_features, Mfcc_labels,
#     test_size=0.3,
#     random_state=42,
#     stratify=Mfcc_labels
# )

# print("训练集标签分布:", Counter(train_labels))
# print("测试集标签分布:", Counter(test_labels))

# # One-hot 编码
# y_train = to_categorical(train_labels, num_classes=2)
# y_test = to_categorical(test_labels, num_classes=2)

# weights = class_weight.compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(train_labels),
#     y=train_labels
# )
# class_weights = dict(enumerate(weights))
# print("类别权重:", class_weights)


# def focal_loss(gamma=2., alpha=0.25):
#     def focal_loss_fixed(y_true, y_pred):
#         epsilon = K.epsilon()
#         y_pred = K.clip(y_pred, epsilon, 1. - epsilon)
#         cross_entropy = -y_true * K.log(y_pred)
#         weight = alpha * K.pow(1 - y_pred, gamma)
#         loss = weight * cross_entropy
#         return K.sum(loss, axis=1)
#     return focal_loss_fixed


# def transformer_encoder_block(inputs, embed_dim, num_heads, ff_dim, rate=0.1):

#     attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads)(inputs, inputs)
#     attn_output = Dropout(rate)(attn_output)
#     out1 = LayerNormalization(epsilon=1e-6)(Add()([inputs, attn_output])) 


#     ffn = Sequential(
#         [Dense(ff_dim, activation="relu"), Dense(embed_dim),]
#     )
#     ffn_output = ffn(out1)
#     ffn_output = Dropout(rate)(ffn_output)
#     out2 = LayerNormalization(epsilon=1e-6)(Add()([out1, ffn_output])) 
#     return out2

# def build_crnn_transformer_model(input_shape=(149, 39, 1), num_classes=2):
#     inputdata = keras.Input(shape=input_shape)
#     x = keras.layers.BatchNormalization()(inputdata)

#     x = keras.layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
#     x = keras.layers.BatchNormalization()(x)
#     x = keras.layers.MaxPooling2D((2,2))(x) 

#     x = keras.layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
#     x = keras.layers.BatchNormalization()(x)
#     x = keras.layers.MaxPooling2D((2,2))(x) 


#     current_shape = K.int_shape(x) 

#     target_shape = (-1, current_shape[3]) 
#     x = keras.layers.Reshape(target_shape)(x)

#     embed_dim = 64       
#     num_heads = 4        
#     ff_dim = 128         
#     transformer_dropout = 0.1


#     print(f"Shape before Transformer: {K.int_shape(x)}")
#     x = transformer_encoder_block(x, embed_dim, num_heads, ff_dim, transformer_dropout)
#     print(f"Shape after Transformer: {K.int_shape(x)}")

#     x = keras.layers.Bidirectional(keras.layers.GRU(64, return_sequences=True, dropout=0.3))(x)

#     x = keras.layers.Bidirectional(keras.layers.GRU(64, return_sequences=False, dropout=0.3))(x)

#     x = keras.layers.Dropout(0.5)(x)
#     x = keras.layers.Dense(32, activation='relu')(x)
#     x = keras.layers.BatchNormalization()(x)
#     outputs = keras.layers.Dense(num_classes, activation='softmax')(x)

#     model = keras.Model(inputs=inputdata, outputs=outputs)
#     optimizer = keras.optimizers.Adam(learning_rate=1e-4)
#     model.compile(loss=focal_loss(gamma=2.0, alpha=0.75), 
#                   optimizer=optimizer,
#                   metrics=['accuracy'])
#     return model

# model = build_crnn_transformer_model(input_shape=(149, 39, 1), num_classes=2)
# # model.summary() 

# early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1) 
# reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7, verbose=1) 

# history = model.fit(
#     train_features, y_train,
#     validation_split=0.2, # 
#     epochs=100,
#     batch_size=64, 
#     class_weight=class_weights,
#     callbacks=[early_stop, reduce_lr] 
# )


# test_loss, test_accuracy = model.evaluate(test_features, y_test, verbose=0)
# print(f"\nTest Loss: {test_loss:.4f}")
# print(f"Test Accuracy: {test_accuracy:.4f}")

# test_predictions = model.predict(test_features)
# y_pred = np.argmax(test_predictions, axis=1)
# y_true = np.argmax(y_test, axis=1)

# print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
# plt.figure(figsize=(6, 5))
# sns.heatmap(confusion_matrix(y_true, y_pred), fmt='g', annot=True, cmap='Blues', cbar=False, 
#             xticklabels=['normal', 'abnormal'],
#             yticklabels=['normal', 'abnormal'])
# plt.title('Confusion Matrix (CRNN+Transformer)')
# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.show()

# print("\nClassification Report:")
# print(classification_report(y_true, y_pred, target_names=['normal', 'abnormal']))


# plt.figure(figsize=(14, 5))

# plt.subplot(1, 2, 1)
# plt.plot(history.history['loss'], label='Training Loss')
# plt.plot(history.history['val_loss'], label='Validation Loss')
# plt.title('Loss Curves')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.grid(True)
# plt.legend()

# plt.subplot(1, 2, 2)
# plt.plot(history.history['accuracy'], label='Training Accuracy')
# plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
# plt.title('Accuracy Curves')
# plt.xlabel('Epochs')
# plt.ylabel('Accuracy')
# plt.grid(True)
# plt.legend()

# plt.tight_layout()
# plt.show()

